# Circuit 5 — Deutsch–Jozsa (3 qubits, balanced oracle)

**What it does:** Determines in a single query whether a black-box function
`f: {0,1}² → {0,1}` is *constant* (same output for all inputs) or *balanced*
(outputs 0 for exactly half the inputs and 1 for the other half). A classical
algorithm needs up to 3 queries in the worst case; Deutsch–Jozsa solves it in 1.

**Circuit structure** (2 input qubits q0, q1 + 1 ancilla q2):
1. X on q2, then H on all 3 qubits — puts ancilla into |−⟩ for phase kickback
2. **Balanced oracle** — CNOT(q0→q2) + CNOT(q1→q2) encodes the balanced function
3. H on input qubits q0, q1 — interference collapses to |11⟩ for balanced, |00⟩ for constant

**All-Clifford:** H, X, CNOT — no stand-in needed. `ManyShotRunner` uses the real
circuit for the heatmap and GIF; `TrajectoryBackend` runs the purity pass.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.backends import TrajectoryBackend, ManyShotRunner
from noisiq.noise import fill_idle_with_identities
from noisiq.visualization import (
    Visualizer,
    export_gif,
    interactive_heatmap,
    plot_error_heatmap,
    per_qubit_purities,
    add_purity_panel,
)

os.makedirs("outputs", exist_ok=True)

profile    = nq.noise.get_hardware("ibm_eagle_r3")
gate_times = profile.gate_times

N_SHOTS      = 2000
N_SHOTS_TRAJ = 400

print(f"noisiq {nq.__version__}")
print(profile.describe())

In [ ]:
def run_purity_pass(circuit, noise_config_twirl, n_qubits, label=""):
    result_traj = TrajectoryBackend().run(
        circuit, noise_model=noise_config_twirl, n_shots=N_SHOTS_TRAJ, seed=42,
    )
    rho = result_traj.final_state
    purities = per_qubit_purities(rho, n_qubits)
    print(f"\n{'─'*50}")
    print(f"Per-qubit purity  [{label}]")
    for q, p in enumerate(purities):
        print(f"  q{q:>2d}  Tr(ρ²) = {p:.4f}  {'█' * int(p * 20)}")
    print(f"{'─'*50}\n")
    return rho, purities


def visualize_circuit(circuit, result_many, noise_pauli, rho, purities,
                      label, gif_name):
    fig = plot_error_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        title=f"IBM Eagle r3 · {label}",
    )
    add_purity_panel(fig, fig.axes[0], rho, circuit.n_qubits)
    plt.show()

    interactive_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        display_mode="annotate",
        title=f"IBM Eagle r3 · {label} [interactive]",
    )
    plt.show()

    viz = Visualizer(circuit)
    viz.many_shot_result = result_many
    gif_path = f"outputs/{gif_name}.gif"
    export_gif(viz, gif_path, purities=purities)
    print(f"GIF → {gif_path}")


print("Helpers ready.")

In [ ]:
N_DJ = 3  # q0, q1 = input qubits; q2 = ancilla


def build_dj_circuit(balanced: bool = True) -> nq.Circuit:
    """
    Deutsch-Jozsa circuit.
    balanced=True  → CNOT(q0→q2) + CNOT(q1→q2) oracle; output |11⟩ on inputs.
    balanced=False → identity oracle (constant-0); output |00⟩ on inputs.
    """
    c = nq.Circuit(
        n_qubits=N_DJ,
        name="dj_balanced" if balanced else "dj_constant",
    )

    # Ancilla q2 into |−⟩: X then H
    c.x(2)
    # Hadamard on all qubits
    for q in range(N_DJ):
        c.h(q)

    # Oracle
    if balanced:
        # Balanced oracle: XOR each input qubit into ancilla via CNOT.
        # Phase kickback flips the phase of states where f(x)=1.
        c.cnot(0, 2)
        c.cnot(1, 2)
    # else: constant-0 oracle → identity, nothing to add.

    # Interference: H on input qubits collapses to |11⟩ (balanced) or |00⟩ (constant)
    c.h(0)
    c.h(1)

    return c


circuit_dj = fill_idle_with_identities(
    build_dj_circuit(balanced=True), gate_times
)
print(f"DJ circuit ops (idle-filled): {len(circuit_dj.operations)}")

In [ ]:
noise_dj = profile.to_noise_model(
    circuit_dj, mode="t2", representation="pauli_twirl",
)

result_dj = ManyShotRunner().run(
    circuit_dj, n_shots=N_SHOTS, noise_config=noise_dj, seed=42,
)
print(f"ManyShotRunner done  zero-error fraction: {result_dj.zero_error_fraction:.4f}")

rho_dj, pur_dj = run_purity_pass(circuit_dj, noise_dj, N_DJ, "Deutsch-Jozsa")

In [ ]:
visualize_circuit(
    circuit_dj, result_dj, noise_dj, rho_dj, pur_dj,
    label="Deutsch-Jozsa (3q, balanced oracle)",
    gif_name="dj_balanced",
)